# Variational Inference for Bayesian Gaussian Mixture Models

Antonio Esteves @ UMinho, Fev 2024

In [ ]:
import numpy             as np
import matplotlib.pyplot as plt
import seaborn           as sns

In [ ]:
class UGMM(object):
    '''Univariate Gaussian Mixture Model with CAVI'''
    def __init__(self, X, K=2, sigma=1):
        self.X = X
        self.K = K
        self.N = self.X.shape[0]
        self.sigma2 = sigma**2

    def _init(self):
        self.phi = np.random.dirichlet([np.random.random()*np.random.randint(1, 10)]*self.K, self.N)
        self.m   = np.random.randint(int(self.X.min()), high=int(self.X.max()), size=self.K).astype(float)
        self.m  += self.X.max()*np.random.random(self.K)
        self.s2  = np.ones(self.K) * np.random.random(self.K)
        print('Initialized the mean')
        print(self.m)
        print('Initialized the variance')
        print(self.s2)

    def get_elbo(self):
        t1  = np.log(self.s2) - self.m/self.sigma2
        t1  = t1.sum()
        t2  = -0.5*np.add.outer(self.X**2, self.s2+self.m**2)
        t2 += np.outer(self.X, self.m)
        t2 -= np.log(self.phi)
        t2 *= self.phi
        t2  = t2.sum()
        return t1 + t2

    def fit(self, max_iter=100, tol=1e-10):
        self._init()
        self.elbo_values = [self.get_elbo()]
        self.m_history   = [self.m]
        self.s2_history  = [self.s2]
        for iter_ in range(1, max_iter+1):
            self._cavi()
            self.m_history.append(self.m)
            self.s2_history.append(self.s2)
            self.elbo_values.append(self.get_elbo())
            if iter_ % 5 == 0:
                print(f'{iter_} Mean={self.m_history[iter_]} Variance={self.s2_history[iter_]}')
            if np.abs(self.elbo_values[-2] - self.elbo_values[-1]) <= tol:
                print(f'ELBO converged with a log likelihood {self.elbo_values[-1] :.3f} at iteration {iter_}')
                break

        if iter_ == max_iter:
            print(f'ELBO ended with a log likelihood {self.elbo_values[-1] :.3f}')


    def _cavi(self):
        self._update_phi()
        self._update_mu_s2()

    def _update_phi(self):
        t1 = np.outer(self.X, self.m)
        t2 = -(0.5*self.m**2 + 0.5*self.s2)
        exponent = t1 + t2[np.newaxis, :]
        self.phi = np.exp(exponent)
        self.phi = self.phi / self.phi.sum(1)[:, np.newaxis]

    def _update_mu_s2(self):
        self.m = (self.phi*self.X[:, np.newaxis]).sum(0) * (1/self.sigma2 + self.phi.sum(0))**(-1)
        assert self.m.size == self.K

        self.s2 = (1/self.sigma2 + self.phi.sum(0))**(-1)
        assert self.s2.size == self.K


In [ ]:
# Generate the K Gaussian means
np.random.seed(4163)

K = 3
mu_arr = np.random.choice(np.arange(-10, 10, 2), K) + np.random.random(K)

print(mu_arr)

In [ ]:
N = 1000

In [ ]:
# draw N samples from each of the K Gaussians (mean=mu_arr[k],sigma=1) 
X = np.random.normal(loc=mu_arr[0], scale=1, size=N)

for _, mu in enumerate(mu_arr[1:]):
    X = np.append(X, np.random.normal(loc=mu, scale=1, size=N))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))

sns.histplot(X[:N],    stat='density', ax=ax, kde=True)
sns.histplot(X[N:N*2], stat='density', ax=ax, kde=True)
sns.histplot(X[N*2:],  stat='density', ax=ax, kde=True)

In [ ]:
ugmm = UGMM(X, 3)
ugmm.fit()

In [ ]:
ugmm.phi.argmax(1)

In [ ]:
sorted(mu_arr)

In [ ]:
sorted(ugmm.m)

In [ ]:
sorted(ugmm.s2)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))

sns.histplot(X[:N],    ax=ax, stat='density', kde=True)
sns.histplot(X[N:N*2], ax=ax, stat='density', kde=True)
sns.histplot(X[N*2:],  ax=ax, stat='density', kde=True)

# Predicted samples for x using mean obtained with CAVI and variance=1.
# The variance is considered to be fixed at 1 in this example, since  
# it was not modeled with a latent variable. Therefore, the s2 generated 
# by CAVI does not aply to 'x', it is the variance of the estimated mean.

sns.kdeplot(np.random.normal(ugmm.m[0], 1, N), ax=ax, fill=False, color='r', linewidth=3)
sns.kdeplot(np.random.normal(ugmm.m[1], 1, N), ax=ax, fill=False, color='r', linewidth=3)
sns.kdeplot(np.random.normal(ugmm.m[2], 1, N), ax=ax, fill=False, color='r', linewidth=3)

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(12, 6))

# Original samples for x
sns.histplot(X[:N],    ax=ax[0], stat='density', kde=True)
sns.histplot(X[N:N*2], ax=ax[0], stat='density', kde=True)
sns.histplot(X[N*2:],  ax=ax[0], stat='density', kde=True)

# Predicted samples for x using mean obtained with CAVI and variance=1.
# The variance is considered to be fixed at 1 in this example, since  
# it was not modeled with a latent variable. Therefore, the s2 generated 
# by CAVI does not aply to 'x', it is the variance of the estimated mean.

sns.kdeplot(np.random.normal(ugmm.m[0], 1.0, N), ax=ax[1], fill=False, color='r')
sns.kdeplot(np.random.normal(ugmm.m[1], 1.0, N), ax=ax[1], fill=False, color='r')
sns.kdeplot(np.random.normal(ugmm.m[2], 1.0, N), ax=ax[1], fill=False, color='r')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 2))
plt.plot(range(len(ugmm.elbo_values)), ugmm.elbo_values, color='red', alpha=0.5)
ax.set_title('ELBO during CAVI optimization')
plt.show()